# Speech Emotion Router (SER): Non-Linear Tabular Modeling & Performance Ceilings

In the previous phase ([`02_baseline_modeling.ipynb`](./02_baseline_modeling.ipynb)), our flat multinomial Logistic Regression benchmark established that static MFCC summaries provide strong demographic separation but struggle significantly with intra-gender emotional ambiguity.

This phase investigates whether non-linear decision boundaries can resolve overlapping affective clusters before we transition to continuous 2D spectral representations.

**Phase Objectives & Experimental Scope:**

* **Non-Linear Tabular Benchmark:** Evaluate a couple non-linear classificaiton models to try and capture the emotional relationships 
  *  For example: Support Vector Classifier with a Radial Basis Function (RBF) kernel on the 26-dimensional static MFCC summary matrix to test the limits of non-linear geometric boundaries.

* **Hyperparameter Calibration:** Conduct a structured grid search across regularization ($C$) and kernel bandwidth ($\gamma$) to balance boundary flexibility against speaker overfitting.

* **Leak-Free Grouped Validation:** Enforce the established 5-fold `StratifiedGroupKFold` harness grouped by `actor_id` with in-pipeline scaling to ensure evaluation on unseen speakers.

* **Performance Ceiling Assessment:** Compare out-of-fold metrics and confusion patterns directly against the linear baseline to determine whether static temporal summaries inherently constrain valence recognition.

In [ ]:
# Standard library
from pathlib import Path
import pprint
from typing import Any, cast

# Data handling and numerical arrays
import joblib
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
from matplotlib.container import BarContainer
import seaborn as sns

# Scikit-learn: pipelines and preprocessing
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Scikit-learn: validation and evaluation metrics
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    ParameterGrid,
    StratifiedGroupKFold,
    cross_val_predict,
    cross_validate,
)

# Scikit-learn: estimators
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Plotting style configuration
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

In [2]:
# Define paths and import the processed feature dataset
project_root = Path.cwd().resolve().parent
data_processed_path = project_root / "data" / "processed"

In [5]:
# Reload artifacts from previous phase

x_path = data_processed_path / "02_1_X_full.parquet"
y_path = data_processed_path / "02_1_Y_full.parquet"
groups_path = data_processed_path / "02_1_groups_full.parquet"

# Audit saved artifacts using .info()
print("--- X Artifact ---")
pd.read_parquet(x_path).info()  # Expected: (1440, 26)

print("\n--- y Artifact ---")
pd.read_parquet(y_path).info()  # Expected: (1440, 1) gender_emotion

print("\n--- Groups Artifact ---")
pd.read_parquet(groups_path).info()  # Expected: (1440, 1) actor_id

--- X Artifact ---
<class 'pandas.DataFrame'>
RangeIndex: 1440 entries, 0 to 1439
Data columns (total 26 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mfcc_mean_0   1440 non-null   float64
 1   mfcc_std_0    1440 non-null   float64
 2   mfcc_mean_1   1440 non-null   float64
 3   mfcc_std_1    1440 non-null   float64
 4   mfcc_mean_2   1440 non-null   float64
 5   mfcc_std_2    1440 non-null   float64
 6   mfcc_mean_3   1440 non-null   float64
 7   mfcc_std_3    1440 non-null   float64
 8   mfcc_mean_4   1440 non-null   float64
 9   mfcc_std_4    1440 non-null   float64
 10  mfcc_mean_5   1440 non-null   float64
 11  mfcc_std_5    1440 non-null   float64
 12  mfcc_mean_6   1440 non-null   float64
 13  mfcc_std_6    1440 non-null   float64
 14  mfcc_mean_7   1440 non-null   float64
 15  mfcc_std_7    1440 non-null   float64
 16  mfcc_mean_8   1440 non-null   float64
 17  mfcc_std_8    1440 non-null   float64
 18  mfcc_mean_9   1440 n